#### Middleware

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

#### Summarization Middleware

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq

model = ChatGroq(model="qwen/qwen3-32b")


agent = create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [3]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [4]:
# Alternative test data 
questions = [
    "what is 2+2?",
    "what is 10*5",
    "what is 100/4",
    "what is 15-7",
    "what is 3*3",
    "what is 4*4",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='de94558e-6570-4bb5-bb8a-5acf7e922089'), AIMessage(content='<think>\nOkay, so I need to figure out what 2+2 is. Let me start by recalling basic arithmetic. Addition is one of the first math operations I learned. When you add two numbers together, you\'re combining their quantities. So, 2 plus 2... Hmm.\n\nLet me visualize it. If I have two apples and then I get two more apples, how many apples do I have in total? That would be four apples. So, 2 + 2 equals 4. But wait, maybe I should check another way. Let\'s use fingers. Hold up two fingers on one hand and two on the other. Counting them all: 1, 2, 3, 4. Yeah, that\'s four.\n\nIs there any context where 2+2 isn\'t 4? Maybe in some special math cases, like modulo arithmetic or binary? For example, in modulo 3 arithmetic, 2 + 2 would be 1 because 4 mod 3 is 1. But the question doesn\'t specify any such context. In standard arithme

#### Token Size

In [5]:
from langchain_core.tools import tool

@tool
def search_hotels(city:str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",550),
            keep=("tokens",200),
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

def count_token(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4   # 4 chars = 1 token

In [6]:
# Run test
cities = ["Paris","London","Tokyo","New York","Dubai","Singapore"]

for city in cities:
    response = agent.invoke({
        "messages":[HumanMessage(content=f"Find hotels in city {city}")]
    },config=config)

    tokens = count_token(response['messages'])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~144 tokens, 4 messages
[HumanMessage(content='Find hotels in city Paris', additional_kwargs={}, response_metadata={}, id='32dece07-2ce8-4102-aef0-b034bf70e8a9'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to find hotels in Paris. Let me check the available tools. There\'s a function called search_hotels that takes a city parameter. Since the user specified Paris, I need to call this function with the city set to Paris. The function\'s description mentions it returns a long response, so I should make sure to handle that properly. Alright, I\'ll structure the tool call with the city parameter as "Paris" and then use the response to provide detailed information back to the user.\n', 'tool_calls': [{'id': '53cr8h0ex', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 156, 'total_tokens': 283, 'completion_time': 0.2137385

#### Fraction

In [7]:
@tool
def search_hotels(city:str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget stay $75"

agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction",0.002), ## 0.2% = ~256 tokens
            keep=("tokens",0.001),
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

cities = ["Paris","London","Tokyo","New York","Dubai","Singapore"]

for city in cities:
    response = agent.invoke({
        "messages":[HumanMessage(content=f"Find hotels in city {city}")]
    },config=config)

    tokens = count_token(response['messages'])
    fraction = tokens / 128000
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} messages")
    print(response['messages'])

Paris: ~525 tokens (0.4102%), 4 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n<think>\nOkay, let\'s see. The user wants me to act as a Context Extraction Assistant. My task is to extract the highest quality and most relevant context from the conversation history provided. The user provided a single message: "Find hotels in city Paris". \n\nFirst, I need to structure the response into the four sections: SESSION INTENT, SUMMARY, ARTIFACTS, and NEXT STEPS. \n\nFor SESSION INTENT, the primary goal is to find hotels in Paris. The user\'s main request is straightforward here. \n\nIn the SUMMARY section, I need to extract the most important context. Since there\'s only one message, the key information is the user\'s request to find hotels in Paris. There\'s no additional context like preferences or constraints mentioned, so I\'ll note that.\n\nARTIFACTS: The user hasn\'t mentioned any files or resources being created or modified, so I should state "None" he

#### Human in the Loop

In [8]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

def read_email_tool(email_id:str) -> str:
    """Mock function to read an email by its ID"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email sen to {recipient} with subject {subject}"

In [9]:
agent = create_agent(
    model=model,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool": False,
            }
        )
    ]
)

In [11]:
config = {"configurable": {"thread_id": "test-approve"}}

response = agent.invoke({
        "messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]
    },config=config)

response

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='c18b2ff1-11c8-41d8-b516-e8ea2646f51a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three: recipient is john@test.com, subject is Hello, body is How are you?. So I need to call send_email_tool with these arguments. No issues here. Just structure the JSON with those parameters.\n", 'tool_calls': [{'id': 'a58fa70ms', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 148, 'prom

In [12]:
from langgraph.types import Command

if "__interrupt__" in response:
    print("Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! Approving...
Result: The email has been successfully sent to john@test.com with the subject "Hello". Let me know if you need anything else!


In [13]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='c18b2ff1-11c8-41d8-b516-e8ea2646f51a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three: recipient is john@test.com, subject is Hello, body is How are you?. So I need to call send_email_tool with these arguments. No issues here. Just structure the JSON with those parameters.\n", 'tool_calls': [{'id': 'a58fa70ms', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 148, 'prom